# Figure 1D, 1E, 1G, Figure S1 & Table S1: Hierarchical RL Modeling (EMU Cohort)

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig 1D (integrated BIC across 6 models), Fig 1E (RW alpha/beta joint density), Fig 1G (binomial glmer fixed effects: RPE weight=0.36, p=0.023; alpha weight=-0.69, p=0.04), Fig S1 (parameter recovery, N=100 x 80 trials, 3 starts), Table S1 |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/work/qasims01/MemoryBanditData/EMU/{learn_df_RWH.csv,full_df_RWH.csv}` |
| **Outputs** | `results/hierarchical_pyem/redesign_v2/{model_comparison_delta_bic.pdf,rw_alpha_beta_joint.pdf,redesign_parameter_recovery.pdf,glmer_hit_coefficients.pdf,combined_df_lme4_hit_rpe_new.csv}`; updated `learn_df_RWH_v2.csv`, `full_df_RWH_v2.csv` |
| **Dependencies** | `membandit`, `rlmodels_k99` (vendored), `pyem` (pinned commit `c0d09807`), R `lme4` (`R/glmer_recognition_emu.R`) |
| **Expected runtime** | ~1-2 hours for full recovery + 6-model EM fits (instant from saved checkpoints/CSVs) |
| **Upstream steps** | Run `Rscript R/glmer_recognition_emu.R` after the recognition export cell and before the Fig 1G plot cell |


## Six-Model Hierarchical Comparison Set

**Candidate models:** `wsls`, `rw`, `rwf`, `vkf_omega`, `bayes_fixed_reward`, `rw_asym`.

Outputs are written to `results/hierarchical_pyem/redesign_v2/`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# Pick up source edits without restarting the kernel.
for _k in list(sys.modules):
    if _k == "rlmodels_k99" or _k.startswith("rlmodels_k99."):
        del sys.modules[_k]

from rlmodels_k99.parameters import MODEL_PARAM_NAMES, build_params
from rlmodels_k99.registry import (
    PRIMARY_MODEL_ORDER,
    SENSITIVITY_MODEL_ORDER,
    REDESIGN_BENCHMARK_ORDER,
    get_model,
)
from rlmodels_k99.recovery import (
    run_redesign_benchmark,
    summarize_parameter_recovery,
    eligible_models_for_real_data,
    compute_finalist_bic,
    likelihood_profile_2d,
    run_model_recovery,
    summarize_model_recovery_rounds,
)
from rlmodels_k99.behavior import (
    attach_rw_rpe_new,
    attach_rw_subject_params,
    load_behavior_tables,
    preprocess_combined_df,
    preprocess_learn_df,
    trim_combined_df_for_lme4,
)
from rlmodels_k99.data import learn_df_to_pyem
from rlmodels_k99.estimation import fit_hierarchical, production_recovery_fit_kwargs
from pyem.utils.math import alpha2norm, beta2norm
from pyem.core.posterior import parameter_recovery as pyem_parameter_recovery

# --- Configuration ---
N_SUBJECTS = 100
NTRIALS = 80
SEEDS = (2,)
N_STARTS = 3
BIC_NSAMPLES = 500
RESUME = True
SHOW_PLOTS = True
PUB_DPI = 300

# Manuscript eligibility gates (pyEM parameter_recovery r + convergence).
MIN_PARAM_R = 0.5
MIN_CONVERGENCE = 0.8

from membandit import paths
LOAD_DIR = Path(paths.get_emu_behavior_dir())
OUT_DIR = Path(paths.results_root()) / "hierarchical_pyem" / "redesign_v2"
CKPT_DIR = OUT_DIR / "checkpoints"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": PUB_DPI,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

MODELS = list(PRIMARY_MODEL_ORDER)
print({"OUT_DIR": str(OUT_DIR), "models": MODELS, "primary": list(PRIMARY_MODEL_ORDER), "n_subjects": N_SUBJECTS, "seeds": SEEDS})


## Model inventory


In [ ]:
print("Primary")
for name in PRIMARY_MODEL_ORDER:
    print(f"  {name}: {MODEL_PARAM_NAMES[name]} — {get_model(name).desc}")
print("Sensitivity")
for name in SENSITIVITY_MODEL_ORDER:
    print(f"  {name}: {MODEL_PARAM_NAMES[name]} — {get_model(name).desc}")
print("Benchmark set")
for name in MODELS:
    print(f"  {name}: {MODEL_PARAM_NAMES[name]}")


## Model descriptions (manuscript)

All models generate binary choices on a two-armed reversal bandit. Unless noted, action probabilities use a softmax over action values, \(P(a)\propto\exp(\beta Q_a)\), with inverse temperature \(\beta\).

### Win-stay / lose-shift (`wsls`)
Free parameter: \(\epsilon\in(0,1)\).

- Trial 0: \(P(a=1)=0.5\).
- After a win (\(r_{t-1}=1\)): \(P(\text{repeat }a_{t-1})=1-\epsilon/2\).
- After a loss (\(r_{t-1}=0\)): \(P(\text{repeat }a_{t-1})=\epsilon/2\).

**Cognitive interpretation.** A one-trial heuristic that stays after rewards and shifts after non-rewards, with \(\epsilon\) controlling residual stochasticity. It is a non-associative baseline rather than a value-learning account.

### Rescorla–Wagner (`rw`)
Free parameters: \(\beta\), \(\alpha\).

\[
Q\leftarrow[0.5,0.5],\quad
\delta=r-Q_a,\quad
Q_a\leftarrow Q_a+\alpha\delta
\]

**Cognitive interpretation.** Classic incremental value learning: only the chosen option is updated by a signed prediction error scaled by a constant learning rate. \(\alpha\) indexes plasticity; \(\beta\) indexes how deterministically values drive choice.

### Fictive Rescorla–Wagner (`rwf`)
Free parameters: \(\beta\), \(\alpha\).

\[
\delta=r-Q_a,\quad
Q_a\leftarrow Q_a+\alpha\delta,\quad
Q_{1-a}\leftarrow Q_{1-a}-\alpha\delta
\]

**Cognitive interpretation.** Counterfactual / fictive updating: the unchosen option is adjusted by the opposite of the experienced PE. Relative preferences therefore move twice as fast as in standard RW for the same \(\alpha\), capturing inferred opportunity costs without observing the forgone outcome.

### Valence-asymmetric RW (`rw_asym`)
Free parameters: \(\beta\), \(d\). Mean learning rate is fixed at \(\bar\alpha=0.35\) (\(c=\mathrm{logit}(0.35)\)).

\[
\alpha_+=\sigma(c+d),\qquad \alpha_-=\sigma(c-d)
\]

Chosen-arm updates use \(\alpha_+\) when \(\delta\ge 0\) and \(\alpha_-\) when \(\delta<0\). When \(d=0\), both rates equal \(0.35\) and the likelihood nests symmetric RW at that fixed rate.

**Cognitive interpretation.** Separate sensitivity to better- vs worse-than-expected outcomes (gain/loss asymmetry) while holding overall learning speed fixed by the task-informed mean rate.

### Bayesian change-point filter (`bayes_fixed_reward`)
Free parameters: \(\beta\), \(p_{\mathrm{switch}}\). Emissions are fixed to the task: \(p(r=1\mid\text{correct})=0.8\), \(p(r=1\mid\text{incorrect})=0.2\).

The agent maintains a posterior over the currently rewarded arm and updates with a Markov switch hazard \(p_{\mathrm{switch}}\); choices are softmax over expected values derived from that posterior.

**Cognitive interpretation.** An ideal observer that knows the reward contingencies but must infer hidden reversals. \(p_{\mathrm{switch}}\) is the subjective hazard of environmental change.

### Volatility Kalman filter (`vkf_omega`)
Free parameters: \(\beta\), \(\omega\). Fixed: \(\lambda=0.2\), \(v_0=1\); initial uncertainty \(W_0=\omega\).

A reduced volatility Kalman filter tracks mean reward beliefs with precision that adapts to inferred volatility; \(\omega\) controls how quickly volatility (and therefore learning gain) changes.

**Cognitive interpretation.** Learning rate is not constant: subjects upweight recent outcomes when the environment is inferred to be volatile and downweight them when it is stable.


## Methods (manuscript)

### Hierarchical fitting (pyEM)
Subject parameters were estimated with hierarchical expectation–maximization (EM) under a Laplace approximation ([pyEM](https://github.com/shawnrhoads/pyEM)). Free parameters live on unconstrained latent scales and are mapped to natural scales with standard link functions (e.g. `norm2alpha` for rates in \((0,1)\), `norm2beta` for \(\beta\in(0,20]\)). At each EM iteration, subject-level maximum a posteriori (MAP) estimates are obtained given the current group prior; the group mean and covariance are then updated from the Laplace posteriors (empirical Bayes).

**Informed initial group prior.** pyEM natively accepts `prior_mu` / `prior_sigma` overrides. That supplied prior is used **only until the first valid M-step**; afterwards the empirical-Bayes group update replaces it. We initialize at task-plausible registry moments (e.g. natural \(\alpha\approx 0.35\), \(\beta\approx 4\), mapped to the latent scale) rather than the generic default. This stabilizes the first E-step (fewer out-of-bounds `1e7` fallbacks) without constraining the final fitted group distribution.

**EM-level multi-start.** pyEM's built-in `max_restarts` only restarts *subject-level* MAP optimizations within an E-step. It does not protect against the EM loop itself converging to a local optimum of the non-convex marginal objective. We therefore restart the full EM from `n_starts = 3` seeds (on top of pyEM's inner restarts) and retain the run with the lowest mean negative posterior likelihood among gate-checked fits. This is a strict superset of stock pyEM behavior.

Convergence and Hessian diagnostics (finite, positive-definite subject Hessians; boundary fraction) were recorded for every fit.

### Integrated BIC model comparison
Models were compared with pyEM's integrated BIC (`BICint` via `EMModel.compute_integrated_bic`). After hierarchical fitting, each subject's marginal likelihood was approximated by Monte Carlo integration of the trial likelihood under the fitted group-level prior (`BIC_NSAMPLES = 500`). Subject evidences were aggregated and penalized by the usual BIC complexity term \(k\log N\) (free parameters \(k\), trials per subject \(N=80\)). Lower `BICint` is better. We report \(\Delta\mathrm{BIC}\) relative to the best model in the primary set (best = 0). Kass–Raftery guidelines for evidence against a worse model: 0–2 weak, 2–6 positive, 6–10 strong, >10 very strong.

### Parameter recovery and eligibility gates
Before empirical fitting, each candidate was subjected to equal-design parameter recovery: \(N=100\) subjects × 80 trials were simulated, then re-estimated with the same pyEM settings used on real data (production budgets, `n_starts = 3`).

**Latent-scale ground truth.** pyEM's EM-MAP estimator assumes a Gaussian population on the unconstrained latent scale — that is the space the optimizer and the group updates operate in. Generating ground-truth subjects as Gaussian draws on that latent scale (`hierarchical_latent`), then mapping to natural parameters with the same link functions used at fit time, therefore tests the estimator under its own generative assumptions, avoids pile-up at natural-scale bounds, and matches empirically observed ranges. Broad natural-scale draws (as in some pyEM README demos) are retained separately as a `stress_natural` robustness mode and are not mixed into the primary recovery claim.

**Manuscript metrics.** Reported recovery statistics use pyEM's validated stock helper `pyem.core.posterior.parameter_recovery` (per-parameter Pearson \(r\) and RMSE on natural-scale true vs estimated subject parameters). A model was eligible for real-data comparison only if **both** of the following held (no gate relaxation):

1. every free parameter had mean \(r > 0.5\) on converged fits;
2. convergence rate \(\ge 0.8\).

Calibration slope and shrinkage ratio were monitored as internal robustness checks only and were **not** used as manuscript eligibility criteria.

Full model-recovery (confusion) analyses are computationally expensive and are reserved for a separate, optional cell below.


## EM parameter recovery

Checkpointed equal-design EM parameter recovery for `MODELS` (primary set + `vkf_w0` head-to-head). Manuscript eligibility uses pyEM `parameter_recovery` Pearson \(r\) and convergence rate only (see Methods). Model recovery / confusion matrices are deferred to the optional cell at the end.


In [ ]:
out = run_redesign_benchmark(
    MODELS,
    seeds=SEEDS,
    n_subjects=N_SUBJECTS,
    ntrials=NTRIALS,
    checkpoint_dir=CKPT_DIR,
    resume=RESUME,
    n_starts=N_STARTS,
    run_identifiability=False,  # EM only; no oracle/MLE/MAP ladder
    # Model recovery is deferred because the full comparison grid is expensive.
    run_model_recovery_flag=False,
)

pr_df = out["parameter_recovery"]
if len(pr_df):
    pr_df.to_csv(OUT_DIR / "parameter_recovery_metrics.csv", index=False)

# Manuscript eligibility: convergence + pyEM parameter_recovery r only.
pr_converged = summarize_parameter_recovery(pr_df, converged_only=True)
pr_all = summarize_parameter_recovery(pr_df, converged_only=False)
rows = []
for model in MODELS:
    sub = pr_converged[pr_converged["model"] == model] if len(pr_converged) else pd.DataFrame()
    all_sub = pr_all[pr_all["model"] == model] if len(pr_all) else pd.DataFrame()
    has_converged_fit = len(sub) > 0
    all_params_ok = has_converged_fit and bool((sub["mean_r"] > MIN_PARAM_R).all())
    convergence_rate = float(all_sub["convergence_rate"].mean()) if len(all_sub) else 0.0
    convergence_ok = convergence_rate >= MIN_CONVERGENCE
    eligible_now = all_params_ok and convergence_ok

    reasons = []
    if not has_converged_fit:
        reasons.append("no_converged_fit")
    elif not all_params_ok:
        bad = sub.loc[sub["mean_r"] <= MIN_PARAM_R, ["param", "mean_r"]]
        reasons.append("param_r_fail:" + ",".join(f"{p}={r:.3f}" for p, r in bad.itertuples(index=False)))
    if not convergence_ok:
        reasons.append(f"convergence_rate={convergence_rate:.3f}")

    rows.append({
        "model": model,
        "worst_param_r": float(sub["mean_r"].min()) if has_converged_fit else np.nan,
        "convergence_rate": convergence_rate,
        # Internal QC only (not manuscript gates); retained for audit.
        "min_calibration_slope": float(sub["mean_slope"].min()) if has_converged_fit else np.nan,
        "min_shrinkage_ratio": float(sub["mean_shrinkage"].min()) if has_converged_fit else np.nan,
        "model_recovery_deferred": True,
        "eligible_for_real_data": eligible_now,
        "promote": eligible_now,
        "reasons": "; ".join(reasons),
    })

promotion_df = pd.DataFrame(rows)
promotion_df.to_csv(OUT_DIR / "parameter_only_eligibility.csv", index=False)
display(promotion_df)

eligible = eligible_models_for_real_data(promotion_df)
print("Temporarily eligible from EM parameter recovery only:", eligible)

# Scatters
def plot_parameter_recovery_pairs(pr_pairs, model_param_names, ncols=3):
    panels = []
    for model in sorted(pr_pairs["model"].unique()):
        subm = pr_pairs[pr_pairs["model"] == model]
        params = [p for p in model_param_names[model] if p in set(subm["param"])]
        for param in params:
            param_rows = subm[subm["param"] == param]
            for seed in sorted(param_rows["seed"].unique()):
                block = param_rows[param_rows["seed"] == seed]
                panels.append((model, param, seed, block))

    if not panels:
        raise ValueError("No parameter-recovery pairs to plot")

    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(3.2 * ncols, 2.8 * nrows),
        squeeze=False,
    )

    for ax, (model, param, seed, block) in zip(axes.flat, panels):
        true_values = pd.to_numeric(block["true"], errors="coerce").to_numpy(dtype=float)
        estimated_values = pd.to_numeric(block["estimated"], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(true_values) & np.isfinite(estimated_values)
        true_valid = true_values[valid]
        estimated_valid = estimated_values[valid]

        ax.scatter(true_valid, estimated_valid, s=12, alpha=0.7)
        if valid.any():
            lo = min(true_valid.min(), estimated_valid.min())
            hi = max(true_valid.max(), estimated_valid.max())
            ax.plot([lo, hi], [lo, hi], "k--", lw=1)

        if len(true_valid) >= 2 and np.std(true_valid) > 0 and np.std(estimated_valid) > 0:
            r = float(np.corrcoef(true_valid, estimated_valid)[0, 1])
            correlation_label = f"r = {r:.2f}"
        else:
            correlation_label = "r = N/A"
        ax.text(
            0.04,
            0.96,
            correlation_label,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
        )
        ax.set_title(f"{model} | {param} | seed {seed}", fontsize=10)

    for ax in axes.flat[len(panels):]:
        ax.set_visible(False)

    fig.tight_layout()
    return fig


pairs_path = CKPT_DIR / "parameter_recovery_pairs.csv"
if pairs_path.exists():
    pr_pairs = pd.read_csv(pairs_path)
    if len(pr_pairs):
        fig = plot_parameter_recovery_pairs(pr_pairs, MODEL_PARAM_NAMES)
        fig.savefig(
            OUT_DIR / "redesign_parameter_recovery.pdf",
            dpi=PUB_DPI,
            bbox_inches="tight",
        )
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)

# One NLL profile for the first multi-param model
for model in MODELS:
    names = MODEL_PARAM_NAMES[model]
    if len(names) < 2:
        continue
    _, _, true = build_params(names, 1, seed=0, model_name=model)
    sim = get_model(model).sim(true, ntrials=min(NTRIALS, 40), seed=1)
    prof = likelihood_profile_2d(model, sim["choices"][0], sim["rewards"][0], true[0], 0, 1, n_grid=15)
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    cs = ax.contourf(prof["x"], prof["y"], prof["nll"], levels=20)
    ax.plot(prof["true_latent"][0], prof["true_latent"][1], "r*", ms=12)
    ax.set_xlabel(prof["param_x"]); ax.set_ylabel(prof["param_y"])
    ax.set_title(f"NLL profile: {model}")
    fig.colorbar(cs, ax=ax, label="NLL")
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"redesign_profile_{model}.pdf", dpi=PUB_DPI)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    break

print("Artifacts:", OUT_DIR)


## Manuscript parameter recovery (pyEM stock helper)

Report per-parameter Pearson \(r\) and RMSE using `pyem.core.posterior.parameter_recovery` on the saved true/estimated subject pairs. These are the manuscript-facing recovery numbers; eligibility above already used the same \(r\) threshold.


In [ ]:
# Manuscript recovery table via pyEM's validated stock helper (no refit).
pairs_path = CKPT_DIR / "parameter_recovery_pairs.csv"
metrics_path = OUT_DIR / "parameter_recovery_metrics.csv"
if not pairs_path.exists():
    print(f"skip stock recovery table: missing {pairs_path}")
else:
    pr_pairs = pd.read_csv(pairs_path)
    seed_pref = int(SEEDS[0]) if len(SEEDS) else int(pr_pairs["seed"].iloc[0])
    rows = []
    for model in MODELS:
        sub = pr_pairs.loc[(pr_pairs["model"] == model) & (pr_pairs["seed"] == seed_pref)]
        if len(sub) == 0:
            sub = pr_pairs.loc[pr_pairs["model"] == model]
        if len(sub) == 0:
            continue
        params = [p for p in MODEL_PARAM_NAMES[model] if p in set(sub["param"])]
        true_cols, est_cols = [], []
        n_subj = None
        for p in params:
            block = sub.loc[sub["param"] == p].sort_values("subject")
            true_cols.append(block["true"].to_numpy(dtype=float))
            est_cols.append(block["estimated"].to_numpy(dtype=float))
            n_subj = len(block) if n_subj is None else n_subj
            assert len(block) == n_subj
        true_mat = np.column_stack(true_cols)
        est_mat = np.column_stack(est_cols)
        rec = pyem_parameter_recovery(true_mat, est_mat)

        if metrics_path.exists():
            mdf = pd.read_csv(metrics_path)
            for j, p in enumerate(params):
                hit = mdf.loc[(mdf["model"] == model) & (mdf["param"] == p) & (mdf["seed"] == seed_pref)]
                if len(hit) == 0:
                    hit = mdf.loc[(mdf["model"] == model) & (mdf["param"] == p)]
                if len(hit):
                    custom_r = float(hit.iloc[0]["correlation"])
                    assert np.isclose(float(rec.corr[j]), custom_r, atol=1e-8), (model, p, rec.corr[j], custom_r)

        for j, p in enumerate(params):
            rows.append(
                {
                    "model": model,
                    "param": p,
                    "seed": int(sub["seed"].iloc[0]),
                    "n_subjects": int(n_subj),
                    "r": float(rec.corr[j]),
                    "rmse": float(rec.rmse[j]),
                    "source": "pyem.core.posterior.parameter_recovery",
                }
            )

    stock_df = pd.DataFrame(rows)
    display(stock_df)
    out_csv = OUT_DIR / "parameter_recovery_stock.csv"
    stock_df.to_csv(out_csv, index=False)
    print("wrote", out_csv)
    if len(stock_df):
        print("worst r by model:")
        print(stock_df.groupby("model")["r"].min().sort_values())


## Fit temporarily eligible models to empirical data

Load `learn_df_RWH.csv`, fit models that passed the temporary EM parameter-recovery gates, and compare by LME (higher better) and integrated BIC (lower better). Model-recovery gating is deferred.


In [ ]:
learn_path = LOAD_DIR / "learn_df_RWH.csv"
combined_path = LOAD_DIR / "full_df_RWH.csv"

if learn_path.exists() and combined_path.exists():
    learn_df, combined_df = load_behavior_tables(LOAD_DIR)
    learn_df = preprocess_learn_df(learn_df)
    combined_df = preprocess_combined_df(combined_df)
    bundle = learn_df_to_pyem(learn_df)
    print(
        f"Loaded+preprocessed learn_df ({learn_path.name}): "
        f"{len(bundle.subject_ids)} good_dm subjects, {bundle.n_trials} trials"
    )
    print(
        f"Loaded+preprocessed combined_df ({combined_path.name}): "
        f"{len(combined_df)} rows, {combined_df['participant'].nunique()} subjects; "
        f"cols include {[c for c in ('zrpe','rpe_cat','phit','Q') if c in combined_df.columns]}"
    )
elif learn_path.exists():
    learn_df = preprocess_learn_df(pd.read_csv(learn_path))
    combined_df = None
    bundle = learn_df_to_pyem(learn_df)
    print(f"Loaded+preprocessed learn_df only: {len(bundle.subject_ids)} subjects")
    print(f"No combined_df at {combined_path}; skipping recognition glmer export.")
else:
    learn_df = None
    combined_df = None
    bundle = None
    print(f"No learn_df at {learn_path}; skipping real-data fit.")


In [ ]:
real_fit_df = pd.DataFrame()
if bundle is None:
    print("skip real-data fit (no data)")
elif not eligible:
    print("skip real-data fit: no models passed promotion gates")
    if len(promotion_df):
        display(promotion_df[["model", "promote", "reasons"]])
else:
    rows = []
    for name in eligible:
        print("fitting:", name)
        fit_kw = production_recovery_fit_kwargs(name, seed=0)
        out = compute_finalist_bic(
            name,
            bundle.all_data,
            ntrials=bundle.n_trials,
            fit_kwargs=fit_kw,
            bic_nsamples=BIC_NSAMPLES,
            require_convergence=False,
        )
        rows.append(
            {
                "model": out["model"],
                "LME": out["LME"],
                "BICint": out["BICint"],
                "convergence": out["convergence"],
                "passed_gates": out["passed_gates"],
                "good_hess_frac": out["good_hess_frac"],
                "boundary_frac": out["boundary_frac"],
            }
        )
        params = pd.DataFrame(out["diagnostics"].subject_params, columns=MODEL_PARAM_NAMES[name])
        params.insert(0, "subject", bundle.subject_ids)
        params.to_csv(OUT_DIR / f"real_params_{name}.csv", index=False)
        del out

    real_fit_df = pd.DataFrame(rows).sort_values("LME", ascending=False)
    real_fit_df.to_csv(OUT_DIR / "real_data_model_comparison.csv", index=False)
    display(real_fit_df)


## Model comparison by integrated BIC

Each eligible model is fit hierarchically (pyEM EM-MAP) to the same subjects and the same 80-trial choice sequences. Comparison uses integrated BIC (`BICint`) as described in Methods. Lower is better; bars show \(\Delta\)BIC relative to the best **primary** model.


In [ ]:
# Publication ΔBIC bar plot (primary models only; winning model highlighted).
cmp_path = OUT_DIR / "real_data_model_comparison.csv"
if "real_fit_df" in globals() and real_fit_df is not None and len(real_fit_df):
    bic_src = real_fit_df.copy()
elif cmp_path.exists():
    bic_src = pd.read_csv(cmp_path)
    print(f"loaded BICint table from {cmp_path}")
else:
    bic_src = pd.DataFrame()
    print("skip ΔBIC plot: no real_fit_df / comparison CSV")

if len(bic_src):
    primary = list(PRIMARY_MODEL_ORDER)
    bic_plot = (
        bic_src.loc[bic_src["model"].isin(primary)]
        .dropna(subset=["BICint"])
        .copy()
    )
    bic_plot = bic_plot.loc[np.isfinite(bic_plot["BICint"])].copy()
    if len(bic_plot) == 0:
        print("skip ΔBIC plot: no finite BICint after filtering")
    else:
        best = float(bic_plot["BICint"].min())
        bic_plot["delta_BIC"] = bic_plot["BICint"] - best
        bic_plot = bic_plot.sort_values("delta_BIC", ascending=True).reset_index(drop=True)
        assert set(bic_plot["model"]).issubset(set(primary))
        assert np.isclose(bic_plot["delta_BIC"].min(), 0.0)
        assert np.isfinite(bic_plot["delta_BIC"]).all()
        display(bic_plot[["model", "LME", "BICint", "delta_BIC"]])

        fig, ax = plt.subplots(figsize=(7.2, 4.0))
        colors = ["#2f6fed" if i == 0 else "#9aa3af" for i in range(len(bic_plot))]
        bars = ax.bar(bic_plot["model"], bic_plot["delta_BIC"], color=colors, edgecolor="white", width=0.72)
        for bar, val in zip(bars, bic_plot["delta_BIC"]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + max(float(bic_plot["delta_BIC"].max()) * 0.02, 0.4),
                f"{val:.1f}",
                ha="center",
                va="bottom",
                fontsize=10,
            )
        ymax = max(float(bic_plot["delta_BIC"].max()) * 1.18, 12.0)
        for y, label in [(2, "weak"), (6, "positive"), (10, "strong")]:
            if y < ymax:
                ax.axhline(y, color="#c7ccd1", ls="--", lw=1.0, zorder=0)
                ax.text(len(bic_plot) - 0.45, y + 0.15, label, ha="right", va="bottom", fontsize=8, color="#6b7280")
        ax.set_ylabel(r"$\Delta$BIC (integrated; best = 0)")
        ax.set_xlabel("model")
        ax.set_title("Hierarchical model comparison (integrated $\Delta$BIC)")
        ax.set_ylim(0, ymax)
        fig.tight_layout()

        out_csv = OUT_DIR / "model_comparison_delta_bic.csv"
        out_pdf = OUT_DIR / "model_comparison_delta_bic.pdf"
        bic_plot.to_csv(out_csv, index=False)
        fig.savefig(out_pdf, dpi=PUB_DPI, bbox_inches="tight")
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)
        print("wrote", out_csv)
        print("wrote", out_pdf)
        print(f"best model: {bic_plot.iloc[0]['model']} (BICint={bic_plot.iloc[0]['BICint']:.1f})")


## Compare hierarchical `rw` fits to existing `learn_df` estimates

Fit hierarchical `rw` on the same good-DM participants, then compare subject-level `alpha`/`beta` and trial-wise RPE against the reference columns already stored in `learn_df`.


In [ ]:
COMPARE_MODELS = ["rw"]

if bundle is None:
    print("skip learn_df comparison (no data)")
else:
    ref_subj = (
        learn_df.loc[learn_df["participant"].isin(bundle.subject_ids), ["participant", "alpha", "beta"]]
        .drop_duplicates("participant")
        .set_index("participant")
        .reindex(bundle.subject_ids)
    )
    ref_trials = (
        learn_df.loc[
            learn_df["participant"].isin(bundle.subject_ids),
            ["participant", "trials", "rpe", "alpha", "beta"],
        ]
        .sort_values(["participant", "trials"], kind="mergesort")
        .reset_index(drop=True)
    )

    param_rows, trial_frames, fit_store = [], [], {}
    for name in COMPARE_MODELS:
        print(f"fitting hierarchical {name} for learn_df comparison...")
        _em, diag = fit_hierarchical(
            name,
            bundle.all_data,
            fit_kwargs=production_recovery_fit_kwargs(name, seed=0),
            n_starts=N_STARTS,
            require_convergence=False,
        )
        fit_store[name] = diag
        est_df = pd.DataFrame(np.asarray(diag.subject_params, dtype=float), columns=MODEL_PARAM_NAMES[name])
        est_df.insert(0, "participant", bundle.subject_ids)

        for i, pid in enumerate(bundle.subject_ids):
            beta_hat = float(est_df.loc[i, "beta"])
            alpha_hat = float(est_df.loc[i, "alpha"])
            param_rows.append(
                {
                    "participant": pid,
                    "model": name,
                    "alpha_learn_df": float(ref_subj.loc[pid, "alpha"]),
                    "beta_learn_df": float(ref_subj.loc[pid, "beta"]),
                    "alpha_fit": alpha_hat,
                    "beta_fit": beta_hat,
                    "convergence": bool(diag.convergence),
                    "lme": float(diag.lme),
                }
            )
            raw = np.array([beta2norm(beta_hat), alpha2norm(alpha_hat)], dtype=float)
            pe = np.asarray(
                get_model(name).fit(raw, bundle.choices[i], bundle.rewards[i], output="all")["PE"],
                dtype=float,
            )
            trial_frames.append(
                pd.DataFrame(
                    {
                        "participant": pid,
                        "trials": np.arange(1, len(pe) + 1),
                        "model": name,
                        "rpe_fit": pe,
                        "alpha_fit": alpha_hat,
                        "beta_fit": beta_hat,
                    }
                )
            )

    param_cmp = pd.DataFrame(param_rows)
    trial_cmp = pd.concat(trial_frames, ignore_index=True).merge(
        ref_trials, on=["participant", "trials"], how="inner"
    )
    param_cmp.to_csv(OUT_DIR / "learn_df_param_comparison_rw.csv", index=False)
    trial_cmp.to_csv(OUT_DIR / "learn_df_rpe_comparison_rw.csv", index=False)

    # Prefer hierarchical real_params_rw.csv when available (same subjects).
    real_rw_path = OUT_DIR / "real_params_rw.csv"
    if real_rw_path.exists():
        real_rw = pd.read_csv(real_rw_path)
        display(real_rw.head())
    display(param_cmp.head())
    print("wrote", OUT_DIR / "learn_df_param_comparison_rw.csv")
    print("wrote", OUT_DIR / "learn_df_rpe_comparison_rw.csv")


## RPE tercile alignment: `learn_df` vs hierarchical RW

We care less about the raw correlation of continuous RPEs than about whether **within-participant tercile categories** agree. For each participant, independently bin `rpe` (original `learn_df`) and `rpe_fit` (hierarchical RW) into terciles labeled `negative` / `neutral` / `positive` — the same convention as `preprocess_learn_df()`. This tests **relative rank alignment within each participant**, not equality of raw magnitudes or shared numerical thresholds across sources.


In [ ]:
# Within-participant RPE tercile alignment (learn_df rpe vs hierarchical RW rpe_fit).
RPE_CATS = ["negative", "neutral", "positive"]


def _cohen_kappa(y_true, y_pred, labels):
    """Unweighted Cohen's kappa without sklearn."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels = list(labels)
    k = len(labels)
    idx = {lab: i for i, lab in enumerate(labels)}
    cm = np.zeros((k, k), dtype=float)
    for a, b in zip(y_true, y_pred):
        if a in idx and b in idx:
            cm[idx[a], idx[b]] += 1.0
    n = cm.sum()
    if n <= 0:
        return float("nan")
    po = np.trace(cm) / n
    pe = float(np.dot(cm.sum(axis=1), cm.sum(axis=0)) / (n * n))
    if np.isclose(pe, 1.0):
        return float("nan")
    return float((po - pe) / (1.0 - pe))


def _within_subj_terciles(series: pd.Series) -> pd.Series:
    out = pd.Series(index=series.index, dtype="object")
    vals = pd.to_numeric(series, errors="coerce")
    valid = vals.dropna()
    if len(valid) < 3:
        return out
    try:
        cats = pd.qcut(valid, q=3, labels=RPE_CATS, duplicates="drop")
        out.loc[valid.index] = cats.astype(str)
    except Exception:
        return out
    return out


# Prefer in-memory trial_cmp; fall back to saved RW comparison CSV.
align_src = None
if "trial_cmp" in globals() and trial_cmp is not None and len(trial_cmp):
    align_df = trial_cmp.loc[trial_cmp["model"] == "rw"].copy()
    align_src = "trial_cmp"
else:
    _path = OUT_DIR / "learn_df_rpe_comparison_rw.csv"
    if _path.exists():
        align_df = pd.read_csv(_path)
        if "model" in align_df.columns:
            align_df = align_df.loc[align_df["model"] == "rw"].copy()
        align_src = str(_path)
    else:
        align_df = pd.DataFrame()
        print("skip tercile alignment: no trial_cmp / learn_df_rpe_comparison_rw.csv")

if len(align_df):
    need = {"participant", "trials", "rpe", "rpe_fit"}
    missing = need - set(align_df.columns)
    if missing:
        raise ValueError(f"tercile alignment missing columns: {sorted(missing)}")

    align_df = align_df.sort_values(["participant", "trials"], kind="mergesort").reset_index(drop=True)
    align_df["rpe_cat_learn"] = (
        align_df.groupby("participant", group_keys=False)["rpe"].apply(_within_subj_terciles)
    )
    align_df["rpe_cat_rw"] = (
        align_df.groupby("participant", group_keys=False)["rpe_fit"].apply(_within_subj_terciles)
    )
    x = align_df.dropna(subset=["rpe_cat_learn", "rpe_cat_rw"]).copy()
    x["agree"] = x["rpe_cat_learn"].astype(str) == x["rpe_cat_rw"].astype(str)

    n_trials = int(len(x))
    n_subj = int(x["participant"].nunique())
    agreement = float(x["agree"].mean()) if n_trials else float("nan")
    kappa = _cohen_kappa(x["rpe_cat_learn"], x["rpe_cat_rw"], RPE_CATS) if n_trials else float("nan")

    cm = (
        pd.crosstab(x["rpe_cat_learn"], x["rpe_cat_rw"])
        .reindex(index=RPE_CATS, columns=RPE_CATS, fill_value=0)
    )
    cm_norm = cm.div(cm.sum(axis=1).replace(0, np.nan), axis=0)

    by_subj = (
        x.groupby("participant", sort=False)
        .agg(
            n_trials=("agree", "size"),
            agreement=("agree", "mean"),
            n_learn_neg=("rpe_cat_learn", lambda s: int((s == "negative").sum())),
            n_rw_neg=("rpe_cat_rw", lambda s: int((s == "negative").sum())),
        )
        .reset_index()
    )
    by_subj["kappa"] = [
        _cohen_kappa(
            x.loc[x["participant"] == pid, "rpe_cat_learn"],
            x.loc[x["participant"] == pid, "rpe_cat_rw"],
            RPE_CATS,
        )
        for pid in by_subj["participant"]
    ]

    summary = pd.DataFrame(
        [
            {
                "n_trials": n_trials,
                "n_subjects": n_subj,
                "exact_agreement": agreement,
                "cohens_kappa": kappa,
                "subj_agreement_mean": float(by_subj["agreement"].mean()),
                "subj_agreement_median": float(by_subj["agreement"].median()),
                "subj_agreement_min": float(by_subj["agreement"].min()),
                "subj_agreement_max": float(by_subj["agreement"].max()),
                "source": align_src,
            }
        ]
    )
    display(summary)
    display(cm)
    display(cm_norm.round(3))
    print(
        f"exact agreement={agreement:.3%} | Cohen κ={kappa:.3f} | "
        f"n={n_trials} trials / {n_subj} subjects"
    )
    print(
        "per-subject agreement: "
        f"mean={by_subj['agreement'].mean():.3%}, "
        f"median={by_subj['agreement'].median():.3%}, "
        f"range=[{by_subj['agreement'].min():.3%}, {by_subj['agreement'].max():.3%}]"
    )

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    im = ax.imshow(cm_norm.to_numpy(dtype=float), vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(RPE_CATS)))
    ax.set_yticks(range(len(RPE_CATS)))
    ax.set_xticklabels(RPE_CATS)
    ax.set_yticklabels(RPE_CATS)
    ax.set_xlabel("hierarchical RW tercile")
    ax.set_ylabel("learn_df tercile")
    ax.set_title("Within-participant RPE tercile alignment")
    for i in range(len(RPE_CATS)):
        for j in range(len(RPE_CATS)):
            count = int(cm.to_numpy()[i, j])
            frac = cm_norm.to_numpy(dtype=float)[i, j]
            if np.isfinite(frac):
                ax.text(
                    j,
                    i,
                    f"{count}\n({frac:.0%})",
                    ha="center",
                    va="center",
                    fontsize=10,
                    color="white" if frac > 0.55 else "black",
                )
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="row-normalized fraction")
    fig.tight_layout()

    out_summary = OUT_DIR / "rw_rpe_tercile_alignment_summary.csv"
    out_subj = OUT_DIR / "rw_rpe_tercile_alignment_by_subject.csv"
    out_cm = OUT_DIR / "rw_rpe_tercile_alignment_confusion.csv"
    out_pdf = OUT_DIR / "rw_rpe_tercile_alignment.pdf"
    summary.to_csv(out_summary, index=False)
    by_subj.to_csv(out_subj, index=False)
    cm.to_csv(out_cm)
    fig.savefig(out_pdf, dpi=PUB_DPI, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    print("wrote", out_summary)
    print("wrote", out_subj)
    print("wrote", out_cm)
    print("wrote", out_pdf)


In [ ]:
# Export learn_df / full_df with hierarchical RW rpe, rpe_cat, alpha, beta (fitted subjects only).
learn_export_path = LOAD_DIR / "learn_df_RWH_v2.csv"
full_export_path = LOAD_DIR / "full_df_RWH_v2.csv"

rw_src = None
if "align_df" in globals() and align_df is not None and len(align_df) and {"rpe_fit", "rpe_cat_rw"}.issubset(align_df.columns):
    rw_trials = align_df.loc[align_df["model"] == "rw"].copy() if "model" in align_df.columns else align_df.copy()
    rw_src = "align_df"
elif "trial_cmp" in globals() and trial_cmp is not None and len(trial_cmp):
    rw_trials = trial_cmp.loc[trial_cmp["model"] == "rw"].copy()
    rw_trials["rpe_cat_rw"] = rw_trials.groupby("participant", group_keys=False)["rpe_fit"].apply(_within_subj_terciles)
    rw_src = "trial_cmp"
else:
    _path = OUT_DIR / "learn_df_rpe_comparison_rw.csv"
    if _path.exists():
        rw_trials = pd.read_csv(_path)
        if "model" in rw_trials.columns:
            rw_trials = rw_trials.loc[rw_trials["model"] == "rw"].copy()
        rw_trials["rpe_cat_rw"] = rw_trials.groupby("participant", group_keys=False)["rpe_fit"].apply(_within_subj_terciles)
        rw_src = str(_path)
    else:
        rw_trials = pd.DataFrame()

# Subject-level alpha/beta from hierarchical RW.
param_src = None
if "param_cmp" in globals() and param_cmp is not None and len(param_cmp) and {"alpha_fit", "beta_fit"}.issubset(param_cmp.columns):
    rw_params = (
        param_cmp.loc[param_cmp["model"] == "rw", ["participant", "alpha_fit", "beta_fit"]]
        .drop_duplicates("participant", keep="last")
        .rename(columns={"alpha_fit": "alpha", "beta_fit": "beta"})
        .copy()
    )
    param_src = "param_cmp"
elif len(rw_trials) and {"alpha_fit", "beta_fit"}.issubset(rw_trials.columns):
    rw_params = (
        rw_trials[["participant", "alpha_fit", "beta_fit"]]
        .drop_duplicates("participant", keep="last")
        .rename(columns={"alpha_fit": "alpha", "beta_fit": "beta"})
        .copy()
    )
    param_src = f"{rw_src}:trial"
else:
    _ppath = OUT_DIR / "real_params_rw.csv"
    if _ppath.exists():
        rw_params = (
            pd.read_csv(_ppath)
            .rename(columns={"subject": "participant"})[["participant", "alpha", "beta"]]
            .drop_duplicates("participant", keep="last")
            .copy()
        )
        param_src = str(_ppath)
    else:
        rw_params = pd.DataFrame()

if learn_df is None or not len(rw_trials) or not len(rw_params):
    print("skip RWH_v2 export: need learn_df, RW trial fits, and RW subject params")
else:
    need = {"participant", "trials", "rpe_fit", "rpe_cat_rw"}
    missing = need - set(rw_trials.columns)
    if missing:
        raise ValueError(f"RW export missing columns: {sorted(missing)}")

    fit_ids = list(pd.unique(rw_params["participant"]))
    rw_trial_map = (
        rw_trials[["participant", "trials", "rpe_fit", "rpe_cat_rw"]]
        .drop_duplicates(["participant", "trials"], keep="last")
        .rename(columns={"rpe_fit": "rpe", "rpe_cat_rw": "rpe_cat"})
    )

    # --- learn_df_RWH_v2 ---
    learn_df_rwh_v2 = learn_df.loc[learn_df["participant"].isin(fit_ids)].copy()
    learn_df_rwh_v2 = learn_df_rwh_v2.drop(columns=["rpe", "rpe_cat", "alpha", "beta"], errors="ignore")
    learn_df_rwh_v2 = learn_df_rwh_v2.merge(rw_trial_map, on=["participant", "trials"], how="left")
    learn_df_rwh_v2 = learn_df_rwh_v2.merge(rw_params, on="participant", how="left")

    n_miss_rpe = int(learn_df_rwh_v2["rpe"].isna().sum())
    n_miss_ab = int(learn_df_rwh_v2[["alpha", "beta"]].isna().any(axis=1).sum())
    if n_miss_rpe:
        raise ValueError(f"learn_df_RWH_v2 has {n_miss_rpe} trials without RW rpe after merge")
    if n_miss_ab:
        raise ValueError(f"learn_df_RWH_v2 has {n_miss_ab} rows without RW alpha/beta after merge")

    learn_df_rwh_v2.to_csv(learn_export_path, index=False)
    print(
        f"wrote {learn_export_path} | subjects={learn_df_rwh_v2['participant'].nunique()} "
        f"| rows={len(learn_df_rwh_v2)} | trial_src={rw_src} | param_src={param_src}"
    )

    # --- full_df_RWH_v2 (combined recognition+learning table) ---
    if "combined_df" in globals() and combined_df is not None and len(combined_df):
        full_src_df = combined_df
        full_src_name = "combined_df"
    else:
        _fpath = LOAD_DIR / "full_df_RWH.csv"
        if _fpath.exists():
            full_src_df = pd.read_csv(_fpath)
            full_src_name = str(_fpath)
        else:
            full_src_df = None
            full_src_name = None

    if full_src_df is None:
        print("skip full_df_RWH_v2 export: no combined_df / full_df_RWH.csv")
    else:
        if "trials_gamble" not in full_src_df.columns:
            raise ValueError("full_df export requires trials_gamble column")

        full_df_rwh_v2 = full_src_df.loc[full_src_df["participant"].isin(fit_ids)].copy()
        full_df_rwh_v2 = full_df_rwh_v2.drop(columns=["rpe", "rpe_cat", "alpha", "beta"], errors="ignore")

        rw_full_map = rw_trial_map.rename(columns={"trials": "trials_gamble"}).copy()
        rw_full_map["trials_gamble"] = pd.to_numeric(rw_full_map["trials_gamble"], errors="coerce")
        full_df_rwh_v2["trials_gamble"] = pd.to_numeric(full_df_rwh_v2["trials_gamble"], errors="coerce")

        full_df_rwh_v2 = full_df_rwh_v2.merge(rw_full_map, on=["participant", "trials_gamble"], how="left")
        full_df_rwh_v2 = full_df_rwh_v2.merge(rw_params, on="participant", how="left")

        n_miss_ab_full = int(full_df_rwh_v2[["alpha", "beta"]].isna().any(axis=1).sum())
        if n_miss_ab_full:
            raise ValueError(f"full_df_RWH_v2 has {n_miss_ab_full} rows without RW alpha/beta after merge")

        # Memory-only rows legitimately lack trials_gamble/rpe; gamble-linked rows should match.
        gamble_rows = full_df_rwh_v2["trials_gamble"].notna()
        n_miss_rpe_full = int(full_df_rwh_v2.loc[gamble_rows, "rpe"].isna().sum())
        if n_miss_rpe_full:
            raise ValueError(
                f"full_df_RWH_v2 has {n_miss_rpe_full} gamble-linked rows without RW rpe after merge"
            )

        full_df_rwh_v2.to_csv(full_export_path, index=False)
        print(
            f"wrote {full_export_path} | subjects={full_df_rwh_v2['participant'].nunique()} "
            f"| rows={len(full_df_rwh_v2)} | source={full_src_name} | "
            f"gamble_rpe_rows={int(gamble_rows.sum())}"
        )


## Publication figure: RW α–β joint distribution

Joint density of hierarchical Rescorla–Wagner subject parameters (learning rate vs inverse temperature).


In [ ]:
# Prefer empirical hierarchical RW fits; fall back to learn_df comparison output.
rw_params = None
real_rw_path = OUT_DIR / "real_params_rw.csv"
if real_rw_path.exists():
    rw_params = pd.read_csv(real_rw_path)[["alpha", "beta"]].dropna()
    src_label = "hierarchical rw (real_params_rw)"
elif "param_cmp" in globals() and param_cmp is not None and len(param_cmp):
    rw_params = (
        param_cmp.loc[param_cmp["model"] == "rw", ["alpha_fit", "beta_fit"]]
        .rename(columns={"alpha_fit": "alpha", "beta_fit": "beta"})
        .dropna()
    )
    src_label = "hierarchical rw (param_cmp)"
else:
    print("skip α–β joint plot: no RW parameters available")

if rw_params is not None and len(rw_params):
    g = sns.jointplot(
        data=rw_params,
        x="alpha",
        y="beta",
        kind="kde",
        fill=True,
        cmap="Spectral_r",
        thresh=0.05,
        levels=12,
        height=5.2,
        space=0,
        joint_kws=dict(bw_adjust=0.5),
        marginal_kws=dict(fill=True, color="#b8a9c9", alpha=0.7, bw_adjust=0.5),
    )

    g.ax_joint.set_xlabel(r"RW $\alpha$")
    g.ax_joint.set_ylabel(r"RW $\beta$")
    g.ax_joint.set_xlim(-0.5, 1.5)
    g.ax_joint.set_ylim(-1, 7)
    g.figure.suptitle(r"Joint distribution of RW $\alpha$ and $\beta$", y=1.02)
    if g.ax_joint.collections:
        cbar = g.figure.colorbar(
            g.ax_joint.collections[-1], ax=g.ax_joint, orientation="horizontal", pad=0.12, fraction=0.05
        )
        cbar.set_label("density")
    # Overlay subject points so KDE tails beyond [0, 1] aren't mistaken for data.
    g.ax_joint.scatter(
        rw_params["alpha"],
        rw_params["beta"],
        s=28,
        c="black",
        alpha=0.9,
        zorder=5,
        edgecolors="none",
    )
    out_path = OUT_DIR / "rw_alpha_beta_joint.pdf"
    g.figure.savefig(out_path, dpi=PUB_DPI, bbox_inches="tight")
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(g.figure)
    print(f"wrote {out_path} ({src_label}; n={len(rw_params)})")


In [ ]:
rw_params

## Recognition memory: P(hit) ~ hierarchical RW predictors

Attach hierarchical `rw` trial RPE (`rpe_new` / `zrpe_new`) and subject α, β (`alpha_new` / `beta_new`), **restrict to Day 1 recognition only**, and export a trimmed CSV for the manuscript `lme4::glmer` binomial model. Day 2 rows are excluded. `zpm` is within-participant Gelman-standardized **image memorability** (`DPRIME` / image d′), not subject-level recognition d′.

Run the R block in the next markdown cell after this Python export completes.


In [ ]:
if combined_df is None:
    print("skip recognition export: combined_df not loaded")
elif "trial_cmp" not in globals() or trial_cmp is None or len(trial_cmp) == 0:
    print("skip recognition export: run the learn_df comparison cell first (needs trial_cmp)")
elif "param_cmp" not in globals() or param_cmp is None or len(param_cmp) == 0:
    print("skip recognition export: run the learn_df comparison cell first (needs param_cmp)")
else:
    combined_df = attach_rw_rpe_new(combined_df, trial_cmp, model="rw")
    # Prefer attach_model if available path already used attach_rw; keep recompute_zpm=False via model helper when present.
    try:
        from rlmodels_k99.behavior import attach_model_subject_params
        combined_df = attach_model_subject_params(
            combined_df, param_cmp, model="rw", recompute_zpm=False
        )
    except TypeError:
        combined_df = attach_rw_subject_params(combined_df, param_cmp, model="rw")
    # Drop any subject-collapsed zpm from helpers; rebuild from image DPRIME below.
    if "zpm" in combined_df.columns:
        combined_df = combined_df.drop(columns=["zpm"])

    day1_mask = combined_df["condition"].astype(str).str.strip().isin(["Day1", "day_1", "Day 1"])
    combined_day1 = combined_df.loc[day1_mask].copy()
    # Retain cohort participants (d' > 0.25, n = 19)
    if "dprime" in combined_day1.columns:
        combined_day1 = combined_day1.loc[pd.to_numeric(combined_day1["dprime"], errors="coerce") > 0.25].copy()
    assert combined_day1["participant"].nunique() == 19, f"Expected 19 EMU participants, got {combined_day1['participant'].nunique()}"
    if len(combined_day1) == 0:
        raise ValueError("No Day1 rows found in combined_df after filtering on condition")

    # zpm from image-level DPRIME (EMU convention), within participant — not subject dprime.
    img_col = "DPRIME" if "DPRIME" in combined_day1.columns else None
    if img_col is None:
        raise ValueError("combined_df Day1 missing image-level DPRIME for zpm")
    # Guard: subject-level dprime must not be used as zpm source.
    nuniq_med = float(combined_day1.groupby("participant")[img_col].nunique().median())
    if nuniq_med <= 1:
        raise ValueError(
            f"DPRIME looks subject-constant (median unique={nuniq_med}); "
            "expected image-level memorability"
        )

    combined_day1["zpm"] = np.nan
    for pid, g in combined_day1.groupby("participant", sort=False):
        vals = pd.to_numeric(g[img_col], errors="coerce")
        mu = float(np.nanmean(vals.to_numpy(dtype=float)))
        sd = float(np.nanstd(vals.to_numpy(dtype=float)))
        if np.isfinite(sd) and sd > 0:
            combined_day1.loc[g.index, "zpm"] = (vals - mu) / (2.0 * sd)

    model_df = trim_combined_df_for_lme4(combined_day1, studied_only=True, require_rpe_new=True)
    model_df["hit"] = model_df["hit"].astype(float)
    zpm_nuniq = model_df.groupby("participant")["zpm"].nunique().median()
    print(f"Day1 median within-subj unique zpm={zpm_nuniq}")
    assert zpm_nuniq > 1, "zpm should be trial-varying image memorability"

    needed = {"zrpe_new", "zpm", "zalpha_new", "zbeta_new", "hit", "participant"}
    missing = needed - set(model_df.columns)
    if missing:
        raise ValueError(f"trimmed combined_df missing columns: {sorted(missing)}")
    print(
        f"Day1 subset={len(combined_day1)} rows; "
        f"lme4 trim={len(model_df)} rows; "
        f"subjects={model_df['participant'].nunique()}"
    )
    print("Day1 condition values:", sorted(model_df["condition"].astype(str).unique()) if "condition" in model_df.columns else "n/a")

    lme4_path = OUT_DIR / "combined_df_lme4_hit_rpe_new.csv"
    glmer_csv = OUT_DIR / "glmer_hit_coefficients.csv"
    combined_df.to_csv(OUT_DIR / "combined_df_with_rpe_new.csv", index=False)
    model_df.to_csv(lme4_path, index=False)
    print("wrote", lme4_path)
    print("wrote", OUT_DIR / "combined_df_with_rpe_new.csv")
    print("Next: run the R markdown cell below, then the coefficient-plot cell.")
    print("Expected R output:", glmer_csv)


## R: Day-1 binomial `glmer` (EMU)

Copy/paste into an R session after the Python export above. Writes manuscript coefficients to `glmer_hit_coefficients.csv`. Requires package `lme4` (no `broom`).

```r
library(lme4)

# Run from repository root via: Rscript R/glmer_recognition_emu.R
base <- "results/hierarchical_pyem/redesign_v2"

d <- read.csv(file.path(base, "combined_df_lme4_hit_rpe_new.csv"))

# Day1-only export; zpm should be trial-varying image memorability
stopifnot(median(tapply(d$zpm, d$participant, function(x) length(unique(x)))) > 1)

d <- subset(
  d,
  complete.cases(hit, zrpe_new, zpm, zalpha_new, zbeta_new, participant)
)

m <- glmer(
  hit ~ zrpe_new + zpm + zalpha_new + zbeta_new + (1 | participant),
  data = d,
  family = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa")
)

summary(m)

s <- coef(summary(m))
out <- data.frame(
  term = rownames(s),
  estimate = s[, "Estimate"],
  std.error = s[, "Std. Error"],
  statistic = s[, "z value"],
  p.value = s[, "Pr(>|z|)"],
  row.names = NULL,
  check.names = FALSE
)

write.csv(out, file.path(base, "glmer_hit_coefficients.csv"), row.names = FALSE)
```


## Publication figure: `glmer` coefficients for P(hit)

Forest / half-eye plot of fixed effects from the binomial `lme4::glmer` model. Requires `glmer_hit_coefficients.csv` produced by the R markdown cell above.


In [ ]:
from scipy.stats import norm

SIG_COLOR, NS_COLOR, ALPHA_SIG = "#2f6fed", "#6e6e6e", 0.05
LABELS = {
    "zrpe_new": r"$rpe$",
    "zpm": r"$pm$",
    "zalpha_new": r"RW$\alpha$",
    "zbeta_new": r"RW$\beta$",
    "(Intercept)": "Intercept",
    "Intercept": "Intercept",
}
ORDER = ["zalpha_new", "zpm", "zrpe_new", "zbeta_new"]

glmer_csv = OUT_DIR / "glmer_hit_coefficients.csv"
if not glmer_csv.exists():
    raise FileNotFoundError(
        f"Missing {glmer_csv}. Run the R markdown cell above to fit glmer and write this CSV, "
        "then re-run this cell."
    )

raw = pd.read_csv(glmer_csv)
raw = raw.rename(columns={c: c.lower().replace(".", "_") for c in raw.columns})
if not {"term", "estimate"}.issubset(raw.columns):
    raise ValueError(f"{glmer_csv.name}: need term/estimate columns; got {list(raw.columns)}")
se_col = "std_error" if "std_error" in raw.columns else ("std_err" if "std_err" in raw.columns else None)
p_col = "p_value" if "p_value" in raw.columns else None
if se_col is None:
    raise ValueError(f"{glmer_csv.name}: missing std.error / std_err")

coef_df = pd.DataFrame(
    {
        "name": raw["term"].astype(str),
        "coef": pd.to_numeric(raw["estimate"], errors="coerce"),
        "se": pd.to_numeric(raw[se_col], errors="coerce"),
        "p": pd.to_numeric(raw[p_col], errors="coerce") if p_col else np.nan,
    }
)
src = f"R glmer ({glmer_csv.name})"

rows = []
for name in ORDER:
    hit = coef_df.loc[coef_df["name"] == name]
    if len(hit) == 0:
        continue
    r = hit.iloc[0]
    coef = float(r["coef"])
    se = float(r["se"]) if np.isfinite(r.get("se", np.nan)) else np.nan
    p = float(r["p"]) if "p" in r and np.isfinite(r.get("p", np.nan)) else np.nan
    sig = bool(np.isfinite(p) and p < ALPHA_SIG)
    rows.append(
        {
            "name": name,
            "label": LABELS.get(name, name),
            "coef": coef,
            "se": se,
            "p": p,
            "significant": sig,
            "color": SIG_COLOR if sig else NS_COLOR,
        }
    )
if not rows:
    raise ValueError("no matching fixed effects in glmer coefficient CSV")

x_lo = min(r["coef"] - 3.0 * r["se"] for r in rows if np.isfinite(r["se"]))
x_hi = max(r["coef"] + 3.0 * r["se"] for r in rows if np.isfinite(r["se"]))
x_lo, x_hi = min(x_lo, 0.0), max(x_hi, 0.0)
pad = 0.08 * (x_hi - x_lo if x_hi > x_lo else 1.0)
x_lo, x_hi = x_lo - pad, x_hi + pad

fig, ax = plt.subplots(figsize=(6.5, 3.6))
y_positions = list(range(len(rows)))[::-1]
x_grid = np.linspace(x_lo, x_hi, 500)
for y, row in zip(y_positions, rows):
    if np.isfinite(row["se"]) and row["se"] > 0:
        dens = norm.pdf(x_grid, loc=row["coef"], scale=row["se"])
        dens = dens / dens.max() * 0.35
        ax.fill_between(x_grid, y, y + dens, color=row["color"], alpha=0.35, linewidth=0)
        lo, hi = row["coef"] - 1.96 * row["se"], row["coef"] + 1.96 * row["se"]
        ax.plot([lo, hi], [y, y], color=row["color"], lw=2.0, solid_capstyle="round")
    ax.plot(row["coef"], y, "o", color=row["color"], ms=7, zorder=3)
    ax.text(
        x_lo - 0.02 * (x_hi - x_lo),
        y,
        row["label"],
        ha="right",
        va="center",
        fontsize=12,
        color=row["color"] if row["significant"] else "black",
    )
ax.axvline(0.0, color="black", ls="--", lw=1.0)
ax.set_xlim(x_lo, x_hi)
ax.set_ylim(-0.4, len(rows) - 0.2)
ax.set_yticks([])
ax.set_xlabel("model weight")
ax.set_title(r"$p(\mathrm{hit})\ \sim$ glmer fixed effects", loc="left")
ax.spines["left"].set_visible(False)
fig.tight_layout()
out_path = OUT_DIR / "glmer_hit_coefficients.pdf"
fig.savefig(out_path, dpi=PUB_DPI, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
print("wrote", out_path, f"({src})")
display(pd.DataFrame(rows)[["label", "name", "coef", "se", "p", "significant"]])


## Notes

- Clean recovery writes to `redesign_v2/`; do not resume mixed `redesign_v1/` checkpoints.
- Final manuscript set: `wsls`, `rw`, `rwf`, `vkf_omega`, `bayes_fixed_reward`, `rw_asym` (`rwus_upe` is sensitivity only).
- Publication figures save as PDF at `PUB_DPI=300`.
- Docs: `docs/model-audit/two-parameter-set.md`, `recoverable-redesign.md`, `promotion-gates.md`.
- Recognition `zpm` is image memorability (Gelman within participant), not subject-level d′.
- Manuscript recognition model is R `lme4::glmer` only (see R markdown cell); coefficient plots require the R-written CSV.


## Model recovery and confusion matrix (manuscript methods; optional / expensive)

**Purpose.** Ask whether the six primary models are mutually recoverable: if data are generated from model \(M_g\), is \(M_g\) also selected as the best-fitting model?

**Procedure.**
1. For each generating model \(M_g\) in the primary set, simulate \(N\) subjects × 80 trials from hierarchical-latent parameter draws (same task schedule as the experiment).
2. Fit **all** primary models to each simulated dataset with the same pyEM settings used empirically.
3. For each simulation round, declare a winner by pyEM integrated BIC (`EMModel.compute_integrated_bic`; or LME via `compute_lme` if BIC is unavailable). pyEM has no built-in model-recovery / confusion helper; the outer simulation–refit loop is project scaffolding around those stock evidence computations.
4. Aggregate across rounds into a confusion matrix whose rows are generating models and columns are estimated (winning) models. Cell \((i,j)\) is the fraction of rounds in which data from model \(i\) were best explained by model \(j\).

**Interpretation.** Strong diagonal dominance indicates recoverable model identity. Off-diagonal mass identifies systematic confusions (e.g. nested or near-equivalent accounts). Because this grid is \(K\times K\) model fits per round and is repeated across rounds, the computation is expensive; the implementation cell below is commented out by default.


In [ ]:
# --- OPTIONAL / EXPENSIVE: model recovery + confusion matrix (commented out) ---
# Uncomment to run. Writes model_recovery_confusion.png at PUB_DPI.
#
from rlmodels_k99.recovery import run_model_recovery, summarize_model_recovery_rounds

MR_MODELS = list(PRIMARY_MODEL_ORDER)
MR_N_SUBJECTS = 40          # smaller than parameter-recovery N for cost
MR_NROUNDS = 2
MR_SEED = 0

rounds_df, summary_df = run_model_recovery(
    MR_MODELS,
    n_subjects=MR_N_SUBJECTS,
    ntrials=NTRIALS,
    nrounds=MR_NROUNDS,
    seed=MR_SEED,
    compute_bic=True,
    checkpoint_dir=CKPT_DIR / "model_recovery",
    resume=True,
    fit_kwargs=dict(njobs=1, verbose=0),
)
rounds_df.to_csv(OUT_DIR / "model_recovery_rounds.csv", index=False)
summary_df.to_csv(OUT_DIR / "model_recovery_summary.csv", index=False)

# Confusion = fraction of rounds won by each Estimated model given Simulated.
conf = (
    summary_df.pivot(index="Simulated", columns="Estimated", values="bestbic")
    .reindex(index=MR_MODELS, columns=MR_MODELS)
    .astype(float)
)
conf = conf.div(conf.sum(axis=1).replace(0, np.nan), axis=0)

fig, ax = plt.subplots(figsize=(6.2, 5.4))
im = ax.imshow(conf.to_numpy(dtype=float), vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(MR_MODELS)))
ax.set_yticks(range(len(MR_MODELS)))
ax.set_xticklabels(MR_MODELS, rotation=45, ha="right")
ax.set_yticklabels(MR_MODELS)
ax.set_xlabel("estimated (winning) model")
ax.set_ylabel("simulated (generating) model")
ax.set_title("Model recovery confusion matrix")
for i in range(len(MR_MODELS)):
    for j in range(len(MR_MODELS)):
        val = conf.to_numpy(dtype=float)[i, j]
        if np.isfinite(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9,
                    color="white" if val > 0.55 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="P(win | generate)")
fig.tight_layout()
out_png = OUT_DIR / "model_recovery_confusion.png"
fig.savefig(out_png, dpi=PUB_DPI, bbox_inches="tight")
plt.show()
print("wrote", out_png)

print("model-recovery cell is commented out (expensive). Uncomment to run.")
